# Random Forest regression of tissue stiffness on gene expression

Fits a Random Forest regressor to predict mean spot stiffness from normalized gene expression, evaluates it with 5-fold cross-validation, and reports feature importances and partial dependence for selected genes.

Input is the merged expression + stiffness table produced by `Visium_processing.ipynb`.

Edit the **Parameters** cell below with your own paths/settings, then run all cells.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import matplotlib as mpl
from pathlib import Path

new_rc_params = {'text.usetex': False, "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)

from scipy.stats import linregress
from sklearn.model_selection import cross_val_predict, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay

## Parameters

Set your paths and settings here. This is the only cell you should need to edit.

In [ ]:
# --- Input path ---
# Merged expression + stiffness table (output of Visium_processing.ipynb)
merged_tsv = Path("/g/korbel/olisov/Visium_Linda/stiffness/normalized/merged_11samples_50um_final.tsv")

# --- Output paths ---
importances_tsv = Path("/g/korbel/olisov/Visium_Linda/stiffness/regression/final_RF_11samples_topgenes_um50.tsv")
figures_dir = Path("/g/korbel/olisov/Visium_Linda/paper_figures/11samples")

# --- Feature selection ---
n_genes = 1000                # number of genes to retain, ranked by fewest zero entries
genes_to_force = ["TFF1", "COL12A1"]  # genes of interest always kept, even outside the top n

# Stiffness columns in the input table. 'target_column' is predicted; all are
# excluded from the feature matrix so the model cannot see the target.
stiffness_columns = ['mean', 'min', 'max']
target_column = 'mean'

# --- Model settings ---
rf_params = dict(
    n_estimators=100,
    random_state=0,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
)
n_splits = 5          # folds for cross-validation
random_state = 0      # seed for the CV split

# --- Plot settings ---
fig_width_mm = 43.5   # single-column figure width for the manuscript
dpi = 600
top_n_importance = 5  # number of features shown in the importance barplot

# Genes to plot partial dependence for, with optional per-gene axis limits.
# Set 'ylim'/'xlim' to None to let matplotlib choose automatically.
partial_dependence_features = {
    'TGFBI':   {'ylim': (400, 700), 'xlim': (-0.1, 4.5)},
    'TFF3':    {'ylim': (380, 900), 'xlim': None},
    'TFF1':    {'ylim': (380, 950), 'xlim': (-0.2, 4.5)},
    'PRAP1':   {'ylim': (400, 660), 'xlim': None},
    'GAPDH':   {'ylim': (400, 760), 'xlim': None},
    'COL12A1': {'ylim': (400, 660), 'xlim': None},
}

# Axis ticks/limits for the predicted-vs-observed scatterplot (set to None to autoscale)
scatter_xticks = (2.2, 2.7, 3.2)
scatter_yticks = (1.5, 2, 2.5, 3, 3.5)
scatter_xlim_left = 2.1

# Derived figure dimensions
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8
figures_dir.mkdir(parents=True, exist_ok=True)

## Load data and select features

Genes are ranked by how few zero entries they have across spots, and the top `n_genes` are kept. This favours genes detected in most spots, which are the ones a tree model can use reliably.

In [ ]:
df = pd.read_csv(merged_tsv, sep='\t', index_col=0)

# Rank gene columns by the number of zeros (fewest zeros first)
gene_columns = [c for c in df.columns if c not in stiffness_columns]
zero_counts = (df[gene_columns] == 0).sum()
sorted_columns = zero_counts.sort_values().index

selected = list(sorted_columns[:n_genes])  # top n genes

# Always keep the genes of interest, even if outside the top n
for gene in genes_to_force:
    if gene not in selected:
        if gene in df.columns:
            selected.append(gene)
        else:
            print(f"Warning: '{gene}' not found in the input table, skipping.")

# Keep the selected genes plus the stiffness columns
df1 = df[selected + stiffness_columns]
print(f"{len(selected)} genes selected, {df1.shape[0]} spots")

## Fit the model and cross-validate

In [ ]:
# Split data into features and target
X = df1.drop(stiffness_columns, axis=1)
y = df1[target_column]

# Initialize the Random Forest model
rf_model = RandomForestRegressor(**rf_params)

# Define cross-validation
kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# Perform cross-validation and collect scores
cv_mse_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='neg_mean_squared_error')
cv_r2_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='r2')

# Convert MSE scores to positive values
cv_mse_scores = -cv_mse_scores

# Calculate average MSE, RMSE, and R-squared from cross-validation
avg_mse = np.mean(cv_mse_scores)
avg_rmse = math.sqrt(avg_mse)
avg_r2 = np.mean(cv_r2_scores)

print("Cross-Validation Results:")
print("Average Root Mean Squared Error:", avg_rmse)
print("Average R-squared:", avg_r2)

# Out-of-fold predictions, used for the predicted-vs-observed plot below
y_pred_cv = cross_val_predict(rf_model, X, y, cv=kf)

# Fit the model on the entire dataset (used for feature importance and partial dependence)
rf_model.fit(X, y)

# Feature importance
importances = rf_model.feature_importances_
features = X.columns

# Create a DataFrame for importances and feature names
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})

# Sort the features by importance
important_features = importance_df.sort_values(by='Importance', ascending=False)

## Predicted vs observed stiffness

Predictions are out-of-fold (`cross_val_predict`), so each spot is predicted by a model that did not see it during training.

In [ ]:
# Apply log10 transform
y_log = np.log10(y)
y_pred_log = np.log10(y_pred_cv)

# Create figure and axis
fig, ax = plt.subplots(figsize=(fig_width_inch, fig_width_inch), dpi=dpi)
sns.scatterplot(x=y_pred_log, y=y_log, color='grey', s=3, ax=ax)

# Correlation line
slope, intercept, r_value, p_value, std_err = linregress(y_pred_log, y_log)
x_sorted = np.linspace(y_pred_log.min(), y_pred_log.max(), 500)
y_sorted = intercept + slope * x_sorted
ax.plot(x_sorted, y_sorted, color='red', linewidth=0.5)

# Annotate r and p value
ax.text(0.5, 0.95, f'$r$ = {r_value:.2f}', transform=ax.transAxes,
        ha='center', va='bottom', fontsize=3)
ax.text(0.5, 0.90, f'$p$ = {p_value:.2e}', transform=ax.transAxes,
        ha='center', va='bottom', fontsize=3)

plt.xlabel("log₁₀Predicted", fontsize=3)
plt.ylabel("log₁₀Observed", fontsize=3)
plt.title("Random Forest stiffness prediction", fontsize=3)

if scatter_xticks is not None:
    ax.set_xticks(scatter_xticks)
if scatter_yticks is not None:
    ax.set_yticks(scatter_yticks)
if scatter_xlim_left is not None:
    ax.set_xlim(left=scatter_xlim_left)

ax.tick_params(axis='both', which='major', pad=0.5, width=0.3, labelsize=3)

sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(figures_dir / 'RF_correlation_11.svg', dpi=dpi, bbox_inches='tight')
plt.show()

## Feature importances

In [ ]:
# Drop features that contributed nothing to the model
important_features = important_features[important_features['Importance'] > 0]
important_features.head(20)

In [ ]:
importances_tsv.parent.mkdir(parents=True, exist_ok=True)
important_features.to_csv(importances_tsv, sep='\t', index=False)
print(f"Wrote {len(important_features)} features to {importances_tsv}")

In [ ]:
# Plot the top features for readability
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=dpi)

top_features = important_features.head(top_n_importance)

sns.barplot(data=top_features, x='Importance', y='Feature', palette='viridis_r')
plt.tick_params(axis='both', which='major', pad=0.5, width=0.3, labelsize=3)
plt.xlabel('Feature Importance', fontsize=3)
plt.ylabel('')
plt.title('Random Forest Feature Importances', fontsize=3)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(figures_dir / 'RF_importance.svg', dpi=dpi, bbox_inches='tight')
plt.show()

## Partial dependence

Shows how predicted stiffness changes across the expression range of each gene, averaging over all other features. Genes and per-gene axis limits are configured in the Parameters cell.

In [ ]:
def plot_partial_dependence(feature, ylim=None, xlim=None):
    """Plot and save the partial dependence of predicted stiffness on one gene."""
    if feature not in X.columns:
        print(f"Warning: '{feature}' is not among the selected features, skipping.")
        return

    fig, ax = plt.subplots(figsize=(fig_width_inch, fig_width_inch), dpi=dpi)

    PartialDependenceDisplay.from_estimator(
        rf_model, X, features=[feature], grid_resolution=100, ax=ax
    )

    ax.set_xlabel(f"{feature} expression")
    ax.set_ylabel("Predicted stiffness")
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlim is not None:
        ax.set_xlim(*xlim)

    sns.despine(top=True, right=True, ax=ax)
    plt.savefig(figures_dir / f'partial_dependence_{feature}.svg', bbox_inches='tight')
    plt.show()


for feature, limits in partial_dependence_features.items():
    plot_partial_dependence(feature, ylim=limits.get('ylim'), xlim=limits.get('xlim'))